# CaptionBERT sentence-address rotary screen
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/captionbert-trainer/blob/v0.1.0/notebooks/captionbert_rotary_screen.ipynb)

Select **G4 / RTX PRO 6000 (96 GB)**. This notebook installs tagged release **v0.1.0**
into an isolated environment compatible with Colab Python 3.13. It leaves Colab's
host packages intact. Each code cell is self-contained; run them in order.
The full screen uses four chunks, two epochs, batch 2048, A0–A3, seeds 1 and 2.
A2g is optional. Add the write-enabled Colab secret **HF_TOKEN** for Hub uploads.
Code stays on GitHub; weights and results go to AbstractPhil/captionbert-8192-rotary.


In [ ]:
import os
import pathlib
import subprocess
import sys
import urllib.request

assert (3, 11) <= sys.version_info[:2] <= (3, 13), "Use a Python 3.11–3.13 runtime"
release = "v0.1.0"
env_dir = pathlib.Path("/content/captionbert-env")
if not (env_dir / "bin/python").exists():
    # Colab may omit stdlib ensurepip. virtualenv seeds pip without that module.
    bootstrap = pathlib.Path("/content/captionbert-bootstrap")
    subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),
                    "virtualenv==20.38.0"], check=True)
    bootstrap_env = os.environ.copy()
    bootstrap_env["PYTHONPATH"] = str(bootstrap)
    subprocess.run([sys.executable, "-m", "virtualenv", "--no-download", str(env_dir)],
                   env=bootstrap_env, check=True)
python = str(env_dir / "bin/python")
# Download the three requirements files from the same immutable code tag.
req_dir = pathlib.Path("/content/captionbert-install")
req_dir.mkdir(parents=True, exist_ok=True)
for name in ("requirements.txt", "requirements-cuda.txt", "requirements-colab.txt"):
    url = f"https://raw.githubusercontent.com/AbstractEyes/captionbert-trainer/{release}/{name}"
    urllib.request.urlretrieve(url, req_dir / name)
subprocess.run([python, "-m", "pip", "install", "-r", str(req_dir / "requirements-colab.txt")], check=True)
subprocess.run([python, "-m", "pip", "install", "--no-deps",
                f"git+https://github.com/AbstractEyes/captionbert-trainer.git@{release}"], check=True)
subprocess.run([python, "-m", "pip", "check"], check=True)
subprocess.run([python, "-c", "import sys,torch,captionbert_rotary; print(sys.version); "
                "print(captionbert_rotary.__version__, torch.__version__, torch.version.cuda); "
                "assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"], check=True)


Run a bounded full-width engineering check before downloading data. This is synthetic
and does not measure sentence quality. Both seeds and all five supported arms run.


In [ ]:
import subprocess
import textwrap

python = "/content/captionbert-env/bin/python"
subprocess.run([python, "-c", textwrap.dedent("""
    from captionbert_rotary.cli import smoke
    smoke(batch_size=8, steps=2, memory_limit_gb=20.0,
          output='/content/captionbert-smoke.json')
""")], check=True)


Prepare only the five required chunks, about 4 GB of targets plus captions and token
caches. Tokenization precomputes sentence slots/anchors and reports rows reaching or
exceeding anchor 24. File hashes and exact row counts protect the caption/target join.


In [ ]:
import subprocess
import textwrap

python = "/content/captionbert-env/bin/python"
subprocess.run([python, "-c", textwrap.dedent("""
    from captionbert_rotary.data import prepare
    prepare('/content/captionbert-data', chunks=(0, 1, 2, 3, 61), max_tokens=256)
""")], check=True)


Train the screen. The literal v2 2,000-step warmup is retained, covering most of this
roughly 2,000-step screen. No gradient accumulation changes the 2,048 negative count.
A worst-case 256-token forward/backward/Adam probe runs before each arm. Completed
arms upload in the background with retries; resume restores already shipped runs.
Use the same configuration when resuming. A 4090 always keeps the 20 GB ceiling.


In [ ]:
import os
import subprocess
import textwrap
from google.colab import userdata

python = "/content/captionbert-env/bin/python"
child_env = os.environ.copy()
child_env["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not child_env["HF_TOKEN"]:
    raise RuntimeError("Set the HF_TOKEN Colab secret with access to the result repository")
subprocess.run([python, "-c", textwrap.dedent("""
    import torch
    from captionbert_rotary import ScreenConfig
    from captionbert_rotary.train import run_screen
    assert torch.cuda.is_available()
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    limit = 80.0 if total_gb > 85 else 20.0
    cfg = ScreenConfig(
        arms=('A0', 'A1', 'A2', 'A3'),  # Add 'A2g' only for a separate declared screen.
        seeds=(1, 2), train_chunks=(0, 1, 2, 3), val_chunk=61,
        epochs=2, batch_size=2048, max_tokens=256,
        data_dir='/content/captionbert-data', output_dir='/content/captionbert-screen',
        memory_limit_gb=limit, push=True, benchmark=True,
    )
    run_screen(cfg)
""")], env=child_env, check=True)


In [ ]:
import json
import pathlib

path = pathlib.Path("/content/captionbert-screen/summary.json")
print(json.dumps(json.loads(path.read_text()), indent=2))
